# json_flatten-gh_archive-spark-iceberg

## 1. Overview

## 2. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.remote("sc://spark-connect:15002").getOrCreate()

## 3. Read

In [ ]:
raw = spark.read.json("s3a://landing/gh_archive")
raw.printSchema()

## 4. Transform

In [ ]:
flat = raw.select(
    F.col("id"),
    F.col("type"),
    F.col("actor.login").alias("actor_login"),
    F.col("repo.name").alias("repo_name"),
    F.col("created_at").cast("timestamp").alias("created_at")
)

## 5. Write

In [ ]:
flat.writeTo("lakehouse.silver.gh_events").using("iceberg").createOrReplace()

## 6. Verify

In [ ]:
spark.sql("SELECT type, count(*) AS n FROM lakehouse.silver.gh_events GROUP BY type ORDER BY n DESC").show()